<a href="https://colab.research.google.com/github/Mitochondrion3000/epic-competiton/blob/main/epic_v2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EPIC v2: улучшенная модель поверх бейзлайна InitiationNet

**Задача.** По одной только последовательности ДНК предсказать для каждой позиции whitelist и каждой цепи силу инициации транскрипции
(счётчик 5'-концов csRNA-seq). Оценка по двум метрикам официального скорера:
* **AUPRC** = `average_precision_score(gt > 0, pred)` по всем позициям шаблона;
* **Spearman** = Pearson между *dense*-рангами предсказания и правды, только в позициях с `gt > 0`.

Ноутбук работает и на учебном виде **`jaNemVect1.1`** (*Nematostella*, у него есть ответы теста, поэтому на нём проверяем улучшения),
и на нашем виде **`Ofas_2.0`** (большой молочайный клоп, *Oncopeltus fasciatus*, лидерборд Large Milkweed Bug). Вид переключается одной строкой в ячейке настроек.

### С чем сравниваемся: бейзлайн (архив `epic_solution` с Google Drive)

| модель | данные для оценки | AUPRC | Spearman |
|---|---|---|---|
| динуклеотидный бейзлайн организаторов | Nematostella, официальный тест | 0.0012 | 0.036 |
| наш прошлый пайплайн (dilated CNN, `epic_pipeline_annotated`) | Nematostella, тест, без квантования | 0.043 | 0.379 |
| InitiationNet `wide`, 24k шагов | Nematostella, тест (half1/half2) | 0.119 | 0.375 |
| InitiationNet `big`, 96k шагов (лучший бейзлайн) | Nematostella, тест (half1/half2) | **0.132** | **0.391** |
| InitiationNet `wide`, 96k шагов | **Ofas_2.0**, hold-out контиги train | **0.159** | **0.444** |

### Что меняем и почему

| # | изменение | откуда идея | переключатель в `CFG` |
|---|---|---|---|
| 1 | **Ансамбль** из нескольких моделей (разные сиды) | у бейзлайна одна модель на вид, без усреднения (write-up, раздел 2.10) | `N_SEEDS` |
| 2 | **EMA весов** (экспоненциальное среднее весов во время обучения) | дешёвый и стабильный способ сгладить шум последних шагов; бейзлайн берёт последние веса | `EMA_DECAY` |
| 3 | **Attention на самом грубом уровне** (шаг 256 нт): позиции окна «видят» друг друга целиком | EDA, раздел D: дивергентные пары +/− на 100–300 нт; раздел F: локальный мотив (PWM) даёт лишь 2.5× случайного, основное качество из широкого контекста | `ATTN_LAYERS` |
| 4 | **Дополнительные выходы на каждую реплику** (r1, r2) рядом с основным (сумма реплик) | EDA, раздел B: только 21% позиций видны в обеих репликах; вспомогательная задача заставляет сеть отделять воспроизводимый сигнал от шума | `REPLICATE_HEADS`, `AUX_WEIGHT` |
| 5 | **Сдвиг положительной позиции внутри окна** (jitter) вместо строгого центра | аугментация: модель не привыкает к тому, что сигнал всегда в середине окна | всегда включено |
| 6 | **TTA: несколько сдвинутых нарезок окон** при предсказании (+ обе цепи) | убирает «швы» на стыках окон | `TTA_SHIFTS` |
| 7 | **Правильное чтение BED**: записи — это отрезки до 6 нт с одним счётом | наш прошлый пайплайн брал только первую позицию отрезка и терял ~3% положительных | всегда включено |
| 8 | Остаётся из бейзлайна: 6 входных каналов (ACGT + строчная буква + N), loss только внутри whitelist, Poisson-loss, RC-эквивариантность, rank-power квантование с подбором `alpha` на hold-out, N → 0 | EDA A/C, write-up раздел 1 | `USE_LOWERCASE`, `ALPHAS`, `KEEP_FRAC` |

Каждое изменение можно выключить, а в разделе 9 есть готовая **абляция** (по очереди включаем изменения и смотрим, что каждое даёт).

### Поток данных
```
genome.fa.gz ──► коды 0..8 (ACGT, acgt, N) ─┐
whitelist.train.bed ──► маска «где считать loss» ┼─► окна 8192 нт ─► InitiationNetV2 ─► log-rate (сумма, r1, r2) × 2 цепи
csRNA-r1/r2-train.bed ──► счётчики (разворачиваем отрезки) ┘                    │
                                                                                 ├─► hold-out «tune»: выбор лучшего шага и alpha
template.test.bed ──► порядок позиций сабмита ◄─ ансамбль + TTA ◄───────────────┘   hold-out «report»: честная оценка
                      └─► rank-power(alpha), N → 0 ─► submission.txt.gz
```
**Среда:** Runtime → Change runtime type → GPU (лучше L4 или A100: у них есть bfloat16; на T4 используется float16 с защитой от переполнений).
Для `Ofas_2.0` нужен High-RAM (геном 1.1 Гб).

## 1. Окружение и настройки

In [ ]:
# 1.1 Библиотеки и устройство
import os, re, gzip, json, math, time, copy, random, glob, tarfile, hashlib, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, precision_recall_curve
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    major, _ = torch.cuda.get_device_capability()
    AMP_DTYPE = torch.bfloat16 if major >= 8 else torch.float16   # bfloat16 есть у Ampere и новее (A100, L4); у T4 только float16
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(), "| смешанная точность:", AMP_DTYPE)
else:
    AMP_DTYPE = None
    print("GPU нет: обучение на CPU пригодно только для проверки кода (FAST_DEV)")

def autocast():
    # контекст смешанной точности; на CPU ничего не делает
    return torch.autocast("cuda", dtype=AMP_DTYPE) if DEVICE == "cuda" else contextlib.nullcontext()

In [ ]:
# 1.2 Все настраиваемые параметры в одном месте
SPECIES  = os.environ.get("EPIC_SPECIES", "jaNemVect1.1")        # "jaNemVect1.1" (учебный, есть ответы теста) или "Ofas_2.0" (наш вид)
DATA_DIR = os.environ.get("EPIC_DATA_DIR", "/content/epic")      # куда скачивать и распаковывать архив
OUT_DIR  = os.environ.get("EPIC_OUT_DIR", f"/content/epic_runs/{SPECIES}")   # чекпоинты, предсказания, сабмит
BASE = f"{DATA_DIR}/{SPECIES}"
FAST_DEV = os.environ.get("EPIC_FAST_DEV") == "1"                # крошечные настройки: только проверить, что весь код проходит
os.makedirs(OUT_DIR, exist_ok=True)

CFG = dict(
    # --- данные ---
    WIN=8192,                 # длина окна ДНК на входе модели, нт
    CROP=512,                 # края окна (с каждой стороны), где не считаем loss и не берём предсказания: там мало контекста
    HOLDOUT_FRAC=0.05,        # доля train-whitelist в каждой из двух отложенных групп контигов (tune и report)
    USE_LOWERCASE=True,       # подавать ли модели флаг «строчная буква» (мягкая маска повторов)
    POS_FRAC=0.5,             # доля окон в батче, построенных вокруг положительной позиции
    # --- модель ---
    CH_LOCAL=96,              # каналы ветки базового разрешения
    TOWER=(128, 192, 256, 320),   # каналы уровней башни; каждый уровень уменьшает разрешение в STRIDE раз
    STRIDE=4,
    ATTN_LAYERS=2,            # слоёв self-attention на самом грубом уровне (0 = как в бейзлайне)
    ATTN_HEADS=8,
    REPLICATE_HEADS=True,     # дополнительные выходы на каждую реплику
    AUX_WEIGHT=0.25,          # вес loss-а реплик относительно основного (сумма реплик)
    # --- обучение ---
    STEPS=48000 if SPECIES == "jaNemVect1.1" else 96000,   # шагов на одну модель (у бейзлайна на Ofas_2.0 было 96000)
    BATCH=8,
    LR=2e-3, WEIGHT_DECAY=1e-2, WARMUP=0.05,
    EMA_DECAY=0.999,          # 0 = без EMA
    EVAL_EVERY=4000,          # как часто проверять качество на группе tune
    EVAL_BP=2_000_000,        # сколько нт группы tune брать в быструю проверку во время обучения
    N_SEEDS=2,                # моделей в ансамбле
    # --- предсказание и постобработка ---
    TTA_SHIFTS=2,             # сколько сдвинутых нарезок окон усреднять
    PRED_BATCH=16,
    ALPHAS=(0.0, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8),   # сетка показателя rank-power
    KEEP_FRAC=0.995,          # alpha = наибольшее, при котором AUPRC на tune >= KEEP_FRAC * лучшего
)
if FAST_DEV:
    CFG.update(WIN=1024, CROP=128, CH_LOCAL=16, TOWER=(16, 24, 32, 48), ATTN_HEADS=4, STEPS=60, BATCH=4,
               EVAL_EVERY=30, EVAL_BP=60_000, PRED_BATCH=8)

CFG_HELP = {
    "WIN": "длина входного окна, нт", "CROP": "отбрасываемые края окна, нт", "HOLDOUT_FRAC": "размер каждой отложенной группы",
    "USE_LOWERCASE": "флаг строчной буквы на входе", "POS_FRAC": "доля окон вокруг положительных позиций",
    "CH_LOCAL": "каналы ветки 1 нт", "TOWER": "каналы уровней башни", "STRIDE": "сжатие на уровень",
    "ATTN_LAYERS": "слоёв attention", "ATTN_HEADS": "голов attention", "REPLICATE_HEADS": "выходы на реплики",
    "AUX_WEIGHT": "вес loss реплик", "STEPS": "шагов на модель", "BATCH": "окон в батче", "LR": "макс. скорость обучения",
    "WEIGHT_DECAY": "L2-регуляризация AdamW", "WARMUP": "доля шагов разогрева", "EMA_DECAY": "затухание EMA",
    "EVAL_EVERY": "период проверки", "EVAL_BP": "нт в быстрой проверке", "N_SEEDS": "моделей в ансамбле",
    "TTA_SHIFTS": "сдвигов нарезки при предсказании", "PRED_BATCH": "окон в батче предсказания",
    "ALPHAS": "сетка alpha", "KEEP_FRAC": "допустимая потеря AUPRC при квантовании",
}
print("вид:", SPECIES, "| данные:", BASE, "| результаты:", OUT_DIR, "| FAST_DEV:", FAST_DEV)
display(pd.DataFrame({"значение": {k: str(v) for k, v in CFG.items()}, "что это": CFG_HELP}))

In [ ]:
# 1.3 Скачивание и распаковка архива вида с Zenodo (doi:10.5281/zenodo.22285753), с проверкой MD5
def download_dataset(species, data_dir):
    if os.path.isdir(f"{data_dir}/{species}"):
        print("датасет уже распакован:", f"{data_dir}/{species}"); return
    import requests
    os.makedirs(data_dir, exist_ok=True)
    rec = requests.get("https://zenodo.org/api/records/22285753", timeout=60).json()
    f = [x for x in rec["files"] if x["key"] == f"{species}.tgz"][0]
    dst = f"{data_dir}/{species}.tgz"
    md5 = hashlib.md5()
    with requests.get(f["links"]["self"], stream=True, timeout=120) as r, open(dst, "wb") as fh:
        r.raise_for_status()
        for chunk in tqdm(r.iter_content(1 << 22), total=f["size"] // (1 << 22) + 1, desc=f"скачиваю {species}", unit="×4МБ"):
            fh.write(chunk); md5.update(chunk)
    want = f.get("checksum", "md5:").split(":")[-1]
    assert not want or md5.hexdigest() == want, "MD5 не совпал: архив повреждён, удалите его и скачайте снова"
    with tarfile.open(dst) as tf:
        try: tf.extractall(data_dir, filter="data")
        except TypeError: tf.extractall(data_dir)
    os.remove(dst)   # архив больше не нужен, экономим диск

download_dataset(SPECIES, DATA_DIR)
for p in sorted(glob.glob(f"{BASE}/**/*", recursive=True)):
    if os.path.isfile(p): print(f"{os.path.getsize(p) / 1e6:9.1f} МБ  {os.path.relpath(p, BASE)}")

## 2. Данные

| файл | что берём | где используется |
|---|---|---|
| `genome/genome.fa.gz` | последовательность; **регистр сохраняем**: код 0–3 = ACGT, 4–7 = acgt (повтор), 8 = N | вход модели |
| `for_participants/whitelist.train.bed.gz` | интервалы, где известны метки train | маска loss, выборка окон, hold-out |
| `for_participants/csRNA-r1/r2-train.bed.gz` | счётчики 5'-концов по репликам. **Каждая строка — отрезок `start..end` до 6 нт с одним счётом**, отрезок разворачиваем в позиции | цели обучения |
| `for_participants/template.test.bed.gz` | порядок позиций сабмита: сначала все строки «+», потом те же строки «−», позиции по возрастанию | предсказание теста |
| `ground_truth/`, `scoring/` | ответы теста и маски половин (**только у jaNemVect1.1**) | офлайн-оценка |

In [ ]:
# 2.1 Интервалы whitelist, шаблон теста и геном
FP = f"{BASE}/for_participants"

def read_bed(path, ncols):
    names = ["chrom", "start", "end", "name", "score", "strand"][:ncols]
    return pd.read_csv(path, sep="\t", header=None, usecols=range(ncols), names=names, dtype={"chrom": str, "strand": str})

wl_train = read_bed(f"{FP}/whitelist.train.bed.gz", 3)
tpl_test = read_bed(f"{FP}/template.test.bed.gz", 6)
TRAIN_CONTIGS = list(dict.fromkeys(wl_train.chrom))
TEST_CONTIGS = list(dict.fromkeys(tpl_test.chrom))

LUT = np.full(256, 8, np.uint8)            # любой символ кроме ACGT/acgt -> 8 (N)
for i, b in enumerate("ACGT"):
    LUT[ord(b)] = i; LUT[ord(b.lower())] = i + 4

def read_genome(path, keep):
    # FASTA -> {контиг: uint8-массив кодов}; читаем только нужные контиги
    genome, name, chunks = {}, None, []
    def flush():
        if name in keep: genome[name] = LUT[np.frombuffer(b"".join(chunks), np.uint8)]
    with gzip.open(path, "rb") as fh:
        for line in tqdm(fh, desc="читаю геном", unit=" строк", mininterval=5):
            if line.startswith(b">"):
                flush(); name = line[1:].split()[0].decode(); chunks = []
            elif name in keep:
                chunks.append(line.rstrip())
    flush()
    return genome

genome = read_genome(f"{BASE}/genome/genome.fa.gz", set(TRAIN_CONTIGS) | set(TEST_CONTIGS))
assert all(c in genome for c in TRAIN_CONTIGS + TEST_CONTIGS), "в fasta нет части контигов из whitelist"

IN_WL = {c: np.zeros(len(genome[c]), bool) for c in TRAIN_CONTIGS}   # True = позиция в whitelist.train
for c, s, e in zip(wl_train.chrom, wl_train.start, wl_train.end):
    IN_WL[c][s:e] = True

TEST_LEN = int((tpl_test.end - tpl_test.start).sum())
print(f"train: {len(TRAIN_CONTIGS)} контигов, whitelist {int((wl_train.end - wl_train.start).sum()) / 1e6:.1f} Мб")
print(f"test : {len(TEST_CONTIGS)} контигов, строк в шаблоне {len(tpl_test)}, позиций в сабмите {TEST_LEN:,}")

In [ ]:
# 2.2 Метки: разворачиваем RLE-отрезки BED и объединяем реплики
REP_FILES = sorted(glob.glob(f"{FP}/csRNA-r[0-9]-train.bed.gz"))
N_REP = len(REP_FILES)
REP_COLS = [f"r{i + 1}" for i in range(N_REP)]

def read_counts_bed(path):
    # строка BED = отрезок [start, end) с одинаковым счётом -> по строке на каждую позицию отрезка
    d = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1, 2, 4, 5], names=["chrom", "start", "end", "count", "strand"],
                    dtype={"chrom": str, "strand": str})
    span = (d.end - d.start).values
    first = np.repeat(np.cumsum(span) - span, span)               # индекс начала своего отрезка для каждой развёрнутой позиции
    return pd.DataFrame({"chrom": np.repeat(d.chrom.values, span),
                         "pos": np.repeat(d.start.values, span) + (np.arange(span.sum()) - first),
                         "strand": np.repeat((d.strand.values == "-").astype(np.int8), span),
                         "count": np.repeat(d["count"].values, span).astype(np.float32)}), len(d)

lab = None
for col, path in zip(REP_COLS, REP_FILES):
    r, n_lines = read_counts_bed(path)
    print(f"{os.path.basename(path)}: строк {n_lines:,} -> позиций {len(r):,} (+{len(r) / n_lines - 1:.1%} после разворота отрезков)")
    r = r.rename(columns={"count": col})
    lab = r if lab is None else lab.merge(r, on=["chrom", "pos", "strand"], how="outer")
lab[REP_COLS] = lab[REP_COLS].fillna(0).astype(np.float32)
lab["tot"] = lab[REP_COLS].sum(axis=1)

TRACK = {}   # контиг -> отсортированные позиции, цепь (0 = +, 1 = −), сумма реплик и каждая реплика
for c, d in lab.groupby("chrom", sort=False):
    d = d.sort_values(["pos", "strand"])
    TRACK[c] = dict(pos=d.pos.values.astype(np.int64), strand=d.strand.values.astype(np.int64),
                    tot=d.tot.values.astype(np.float32), reps=d[REP_COLS].values.T.astype(np.float32))

outside = sum(int((~IN_WL[c][t["pos"]]).sum()) for c, t in TRACK.items() if c in IN_WL)
print(f"положительных позиций-цепей: {len(lab):,}; вне whitelist: {outside}; реплик: {N_REP}")
if N_REP < 2 and CFG["REPLICATE_HEADS"]:
    CFG["REPLICATE_HEADS"] = False; print("реплика одна -> REPLICATE_HEADS выключен")

In [ ]:
# 2.3 Паспорт данных в картинках (подробный разбор в epic_eda_explained)
def class_stats(contigs):
    # сколько позиций whitelist и положительных позиций приходится на заглавные, строчные буквы и N
    n_bp, n_pos = np.zeros(3), np.zeros(3)
    for c in contigs:
        cls = np.minimum(genome[c] // 4, 2)                       # 0 = ACGT, 1 = acgt, 2 = N
        n_bp += np.bincount(cls[IN_WL[c]], minlength=3)
        if c in TRACK: n_pos += np.bincount(cls[TRACK[c]["pos"]], minlength=3)
    return n_bp, n_pos

n_bp, n_pos = class_stats(TRAIN_CONTIGS)
test_cls = np.zeros(3)
for c, s, e in zip(tpl_test.chrom, tpl_test.start, tpl_test.end):
    test_cls += np.bincount(np.minimum(genome[c][s:e] // 4, 2), minlength=3)

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
lens = pd.Series({c: len(genome[c]) for c in TRAIN_CONTIGS + TEST_CONTIGS})
role = pd.Series({**{c: "train" for c in TRAIN_CONTIGS}, **{c: "test" for c in TEST_CONTIGS}})
for r_, col in (("train", "tab:blue"), ("test", "tab:red")):
    ax[0, 0].hist(np.log10(lens[role == r_]), bins=40, alpha=.6, color=col, label=f"{r_}: {int((role == r_).sum())} конт., {lens[role == r_].sum() / 1e6:.0f} Мб")
ax[0, 0].set(xlabel="log10(длина контига, нт)", ylabel="контигов", title="Контиги train и test"); ax[0, 0].legend()

w = 0.38; x = np.arange(3)
ax[0, 1].bar(x - w / 2, n_bp / n_bp.sum(), w, label="train whitelist"); ax[0, 1].bar(x + w / 2, test_cls / test_cls.sum(), w, label="test шаблон")
ax[0, 1].set_xticks(x, ["ACGT", "acgt (повтор)", "N"]); ax[0, 1].set(title="Состав позиций", ylabel="доля"); ax[0, 1].legend()

rate = n_pos / np.maximum(2 * n_bp, 1)
ax[0, 2].bar(["ACGT", "acgt", "N"], rate * 1e3, color=["tab:green", "tab:olive", "gray"])
ax[0, 2].set(title="Положительных на 1000 позиций-цепей", ylabel="‰")
for i, v in enumerate(rate * 1e3): ax[0, 2].text(i, v, f"{v:.3f}", ha="center", va="bottom")

bins = [1, 2, 3, 4, 6, 11, 51, np.inf]; labels = ["1", "2", "3", "4-5", "6-10", "11-50", ">50"]
ax[1, 0].bar(labels, pd.cut(lab.tot, bins, right=False).value_counts(normalize=True, sort=False).values)
ax[1, 0].set(title=f"Сумма реплик в положительной позиции (счёт 1: {(lab.tot == 1).mean():.0%})", xlabel="счёт", ylabel="доля позиций")

if N_REP >= 2:
    both = int(((lab.r1 > 0) & (lab.r2 > 0)).sum())
    ax[1, 1].bar(["только r1", "только r2", "обе"], [int(((lab.r1 > 0) & (lab.r2 == 0)).sum()), int(((lab.r2 > 0) & (lab.r1 == 0)).sum()), both],
                 color=["tab:orange", "tab:purple", "tab:green"])
    ax[1, 1].set(title=f"Воспроизводимость: в обеих репликах {both / len(lab):.0%}", ylabel="позиций")

dens = pd.Series({c: len(TRACK.get(c, {"pos": []})["pos"]) / IN_WL[c].sum() * 1e6 for c in TRAIN_CONTIGS if IN_WL[c].sum() > 0})
dens = dens[lens[dens.index].sort_values(ascending=False).index[:40]]
ax[1, 2].bar(range(len(dens)), dens.values); ax[1, 2].set(title="Положительных на Мб whitelist (40 крупнейших train-контигов)", xlabel="контиг (по убыванию длины)")
plt.tight_layout(); plt.show()
print("Как читать: строчные буквы (повторы) обеднены инициацией, но не пусты -> подаём их модели отдельным каналом, а не выбрасываем. "
      "На N инициации почти нет -> в сабмите N получают 0.")

## 3. Отложенные контиги: `tune` и `report`

Тест у настоящих видов закрыт, поэтому для честной оценки откладываем **две непересекающиеся группы train-контигов** (по ~5% whitelist каждая):
* **tune** — выбираем по ней лучший шаг обучения и показатель `alpha` квантования;
* **report** — на неё ни обучение, ни настройка не смотрят; именно эти цифры сравниваем с бейзлайном.

Контиги набираются жадно так, чтобы доля положительных в группе была как можно ближе к средней по виду (обе метрики сильно зависят от этой доли).

In [ ]:
# 3.1 Жадный выбор групп tune и report
wl_bp = (wl_train.assign(bp=wl_train.end - wl_train.start).groupby("chrom", sort=False).bp.sum())
pos_n = pd.Series({c: len(TRACK[c]["pos"]) if c in TRACK else 0 for c in wl_bp.index})
GLOBAL_RATE = pos_n.sum() / (2 * wl_bp.sum())

def pick_group(avail, target_bp):
    sel, s_bp, s_pos = [], 0, 0
    while s_bp < target_bp and avail:
        best = min(avail, key=lambda c: abs((s_pos + pos_n[c]) / (2 * (s_bp + wl_bp[c])) - GLOBAL_RATE))
        sel.append(best); s_bp += wl_bp[best]; s_pos += pos_n[best]; avail.remove(best)
    return sel

min_bp = max(min(100_000, int(wl_bp.max())), int(wl_bp.sum()) // 1000)   # совсем мелкие контиги в hold-out не берём
avail = [c for c in wl_bp.index if wl_bp[c] >= min_bp]
GROUPS = {g: pick_group(avail, CFG["HOLDOUT_FRAC"] * wl_bp.sum()) for g in ("tune", "report")}
FIT_CONTIGS = [c for c in TRAIN_CONTIGS if c not in set(GROUPS["tune"]) | set(GROUPS["report"])]

summary = pd.DataFrame({g: {"контигов": len(cs), "Мб whitelist": wl_bp[cs].sum() / 1e6, "доля положительных": pos_n[cs].sum() / (2 * wl_bp[cs].sum())}
                        for g, cs in [("обучение", FIT_CONTIGS), ("tune", GROUPS["tune"]), ("report", GROUPS["report"])]}).T
display(summary.style.format({"Мб whitelist": "{:.1f}", "доля положительных": "{:.2e}"}))
print("tune  :", GROUPS["tune"][:10], "..." if len(GROUPS["tune"]) > 10 else "")
print("report:", GROUPS["report"][:10], "..." if len(GROUPS["report"]) > 10 else "")

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].pie(summary["Мб whitelist"], labels=summary.index, autopct="%.0f%%", colors=["tab:blue", "tab:orange", "tab:green"])
ax[0].set_title("Как поделён train-whitelist")
ax[1].bar(summary.index, summary["доля положительных"], color=["tab:blue", "tab:orange", "tab:green"])
ax[1].axhline(GLOBAL_RATE, ls="--", color="k", lw=1, label="среднее по виду"); ax[1].legend()
ax[1].set_title("Доля положительных позиций-цепей в группах")
plt.tight_layout(); plt.show()

## 4. Как строятся обучающие окна

* Окно длиной `WIN` берётся из whitelist обучающих контигов. С вероятностью `POS_FRAC` окно строится вокруг случайной положительной позиции,
  причём позиция попадает **в случайное место центральной зоны** (в бейзлайне — строго в центр), иначе окно равномерно случайно по whitelist.
* Цели окна: массив `(K, 2, WIN)`: `K = 1 + число реплик` (канал 0 — сумма реплик, это то, что оценивается), 2 цепи.
* Маска: 1 внутри whitelist, 0 вне него и за краями контига. Loss считается только там, где маска 1, и только в центральной зоне (без `CROP` с краёв).
* Аугментация «обратная цепь» не нужна: модель симметрична по построению (раздел 5).

In [ ]:
# 4.1 Окна и сэмплер
N_TRACKS = 1 + N_REP        # сумма реплик + каждая реплика

def window_codes(c, start, L):
    # коды окна [start, start+L) контига c; за краями контига -> N (8)
    seq = genome[c]; out = np.full(L, 8, np.uint8)
    a, b = max(start, 0), min(start + L, len(seq))
    if b > a: out[a - start:b - start] = seq[a:b]
    return out

def window_targets(c, start, L):
    # цели (N_TRACKS, 2, L) и маска whitelist (L,) для окна
    y = np.zeros((N_TRACKS, 2, L), np.float32); mask = np.zeros(L, np.float32)
    a, b = max(start, 0), min(start + L, len(genome[c]))
    if c in IN_WL and b > a: mask[a - start:b - start] = IN_WL[c][a:b]
    t = TRACK.get(c)
    if t is not None:
        i0, i1 = np.searchsorted(t["pos"], [start, start + L])
        p, s = t["pos"][i0:i1] - start, t["strand"][i0:i1]
        y[0, s, p] = t["tot"][i0:i1]
        for r in range(N_REP): y[1 + r, s, p] = t["reps"][r, i0:i1]
    return y, mask

class WindowSampler:
    def __init__(self, contigs, cfg, seed):
        self.cfg, self.rng = cfg, np.random.default_rng(seed)
        iv = wl_train[wl_train.chrom.isin(set(contigs))]
        self.iv_c, self.iv_s = iv.chrom.values, iv.start.values
        self.iv_len = (iv.end - iv.start).values; self.iv_cum = np.cumsum(self.iv_len)
        have = [c for c in contigs if c in TRACK]
        self.pos_c = np.concatenate([np.full(len(TRACK[c]["pos"]), i) for i, c in enumerate(have)])
        self.pos_p = np.concatenate([TRACK[c]["pos"] for c in have]); self.pos_names = have

    def centres(self, n):
        out = []
        for _ in range(n):
            if self.rng.random() < self.cfg["POS_FRAC"]:            # окно вокруг положительной позиции
                j = self.rng.integers(len(self.pos_p)); out.append((self.pos_names[self.pos_c[j]], int(self.pos_p[j]), True))
            else:                                                    # равномерно по whitelist (вероятность пропорциональна длине)
                u = self.rng.integers(self.iv_cum[-1]); k = np.searchsorted(self.iv_cum, u, side="right")
                out.append((self.iv_c[k], int(self.iv_s[k] + u - (self.iv_cum[k] - self.iv_len[k])), False))
        return out

    def batch(self, n):
        W, C = self.cfg["WIN"], self.cfg["CROP"]
        codes, ys, masks = [], [], []
        for c, p, _ in self.centres(n):
            start = p - C - int(self.rng.integers(W - 2 * C))         # позиция p попадает в случайное место центральной зоны
            codes.append(window_codes(c, start, W)); y, m = window_targets(c, start, W); ys.append(y); masks.append(m)
        return (torch.from_numpy(np.stack(codes)), torch.from_numpy(np.stack(ys)), torch.from_numpy(np.stack(masks)))

# --- картинка: одно окно вокруг положительной позиции ---
smp = WindowSampler(FIT_CONTIGS, CFG, seed=123)
W, C = CFG["WIN"], CFG["CROP"]
c, p, _ = next(x for x in smp.centres(200) if x[2])
start = p - W // 2
codes = window_codes(c, start, W); y, m = window_targets(c, start, W)
xs = np.arange(W)
fig, ax = plt.subplots(3, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [1, 1, 3]})
ax[0].imshow(np.minimum(codes // 4, 2)[None], aspect="auto", cmap=plt.matplotlib.colors.ListedColormap(["#4c9be8", "#f2a541", "#999999"]), vmin=0, vmax=2)
ax[0].set_yticks([]); ax[0].set_title(f"Окно {c}:{start:,}-{start + W:,}.  Полоса: синий ACGT, оранжевый acgt (повтор), серый N")
ax[1].fill_between(xs, m, step="mid", color="tab:green", alpha=.5); ax[1].set_yticks([0, 1]); ax[1].set_ylabel("whitelist")
ax[2].bar(xs, y[0, 0], width=3, color="tab:red", label="+ цепь (сумма реплик)"); ax[2].bar(xs, -y[0, 1], width=3, color="tab:blue", label="− цепь")
for a in ax[1:]: a.axvspan(0, C, color="k", alpha=.08); a.axvspan(W - C, W, color="k", alpha=.08)
ax[2].set(xlabel="позиция в окне", ylabel="счёт (минус вниз)"); ax[2].legend(loc="upper right")
ax[2].text(C / 2, ax[2].get_ylim()[1] * .8, "CROP:\nloss не\nсчитаем", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

# --- сколько положительных позиций в окнах двух типов ---
npos = {True: [], False: []}
for c, p, is_pos in smp.centres(600):
    y, m = window_targets(c, p - W // 2, W); npos[is_pos].append(int((y[0, :, C:-C] > 0).sum()))
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist([npos[True], npos[False]], bins=30, label=["вокруг положительной", "равномерное"], log=True)
ax.set(xlabel="положительных позиций-цепей в центральной зоне окна", ylabel="окон", title="Зачем нужен POS_FRAC"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Модель InitiationNetV2

Основа — InitiationNet из бейзлайна, добавлены attention и выходы на реплики:

1. **Вход** `(B, 6, L)`: one-hot ACGT, флаг строчной буквы, флаг N.
2. **Stem**: свёртка ширины 11 → `CH_LOCAL` каналов.
3. **Локальная ветка** (разрешение 1 нт): 2 остаточных блока (ширина 5, дилатации 1 и 2) — короткие мотивы вокруг точки старта.
4. **Башня** из `len(TOWER)` уровней: каждый уровень сжимает длину в `STRIDE`=4 раза (шаги 4, 16, 64, 256 нт) и применяет 2 остаточных блока.
5. **Новое: attention** на самом грубом уровне (для окна 8192 это 32 «токена» по 256 нт): каждый участок окна видит все остальные,
   так модель может связать, например, промотор на одной цепи с дивергентным стартом на другой.
6. **Слияние**: каждый уровень проецируется в `CH_LOCAL` каналов, растягивается до 1 нт и **складывается** с локальной веткой.
7. **Голова**: 1×1 свёртки → `K` выходов на позицию: log-rate суммы реплик (основной) и **новое:** log-rate каждой реплики.

**Обе цепи одной сетью.** Сеть предсказывает только плюс-цепь; минус-цепь = та же сеть на обратно-комплементарном окне, выход развёрнут.
Значит, модель симметрична к смене цепи *по построению*.

**Функция потерь**: Poisson NLL `exp(λ) − y·λ` в позициях whitelist центральной зоны, отдельно для суммы реплик (вес 1) и для реплик (вес `AUX_WEIGHT`).

In [ ]:
# 5.1 Архитектура
def code_table(device):
    # код 0..8 -> 6 каналов: one-hot ACGT (одинаковый для A и a), флаг строчной буквы, флаг N
    t = torch.zeros(9, 6, device=device)
    for b in range(4):
        t[b, b] = 1; t[b + 4, b] = 1; t[b + 4, 4] = 1
    t[8, 5] = 1
    return t

TABLE = code_table(DEVICE)
RC_PERM = [3, 2, 1, 0, 4, 5]          # A<->T, C<->G; флаги регистра и N от цепи не зависят

def encode(codes, use_lowercase=True):
    x = TABLE[codes.long()].permute(0, 2, 1)                     # (B, 6, L)
    if not use_lowercase: x = x * torch.tensor([1, 1, 1, 1, 0, 1.], device=x.device).view(1, 6, 1)
    return x

def rc(x):
    return x[:, RC_PERM].flip(-1)                               # обратно-комплементарная последовательность

class ResBlock(nn.Module):
    # свёртка -> BN -> GELU -> свёртка -> BN, плюс вход (остаточная связь)
    def __init__(self, ch, k=5, d=1):
        super().__init__()
        self.c1 = nn.Conv1d(ch, ch, k, padding=d * (k // 2), dilation=d); self.n1 = nn.BatchNorm1d(ch)
        self.c2 = nn.Conv1d(ch, ch, k, padding=d * (k // 2), dilation=d); self.n2 = nn.BatchNorm1d(ch)
    def forward(self, x):
        h = F.gelu(self.n1(self.c1(x)))
        return F.gelu(x + self.n2(self.c2(h)))

class AttentionBottleneck(nn.Module):
    # self-attention по «токенам» самого грубого уровня; позиция кодируется depthwise-свёрткой (работает при любой длине окна)
    def __init__(self, ch, n_layers, n_heads):
        super().__init__()
        self.pos = nn.Conv1d(ch, ch, 3, padding=1, groups=ch)
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(ch, n_heads, dim_feedforward=2 * ch, dropout=0.1,
                                                                activation="gelu", batch_first=True, norm_first=True)
                                     for _ in range(n_layers)])
    def forward(self, z):                                       # z: (B, C, T)
        h = (z + self.pos(z)).transpose(1, 2)
        for layer in self.layers: h = layer(h)
        return h.transpose(1, 2)

class InitiationNetV2(nn.Module):
    def __init__(self, ch_local=96, tower=(128, 192, 256, 320), stride=4, attn_layers=2, attn_heads=8, n_out=3, in_ch=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, ch_local, 11, padding=5), nn.BatchNorm1d(ch_local), nn.GELU())
        self.local = nn.Sequential(ResBlock(ch_local, 5, 1), ResBlock(ch_local, 5, 2))
        self.downs, self.blocks, self.ups = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        prev = ch_local
        for c in tower:
            self.downs.append(nn.Sequential(nn.Conv1d(prev, c, 2 * stride + 1, stride=stride, padding=stride), nn.BatchNorm1d(c), nn.GELU()))
            self.blocks.append(nn.Sequential(ResBlock(c, 5, 1), ResBlock(c, 5, 3)))
            self.ups.append(nn.Conv1d(c, ch_local, 1))
            prev = c
        self.attn = AttentionBottleneck(prev, attn_layers, attn_heads) if attn_layers > 0 else nn.Identity()
        self.head = nn.Sequential(nn.Conv1d(ch_local, ch_local, 1), nn.GELU(), nn.Conv1d(ch_local, n_out, 1))
        nn.init.constant_(self.head[-1].bias, -8.0)              # стартовая частота exp(-8) ~ 3e-4 на позицию, порядок реальной доли положительных

    def forward(self, x):                                       # x: (B, 6, L) -> (B, n_out, L) log-rate плюс-цепи
        L = x.shape[-1]
        h = self.local(self.stem(x)); fused, z = h, h
        for i, (down, blk, up) in enumerate(zip(self.downs, self.blocks, self.ups)):
            z = blk(down(z))
            if i == len(self.downs) - 1: z = self.attn(z)
            fused = fused + F.interpolate(up(z), size=L, mode="linear", align_corners=False)
        return self.head(fused)

def forward_both(net, x):
    # (B, K, 2, L): обе цепи одной сетью; минус-цепь = сеть на RC-окне, выход развёрнут обратно
    return torch.stack([net(x), net(rc(x)).flip(-1)], dim=2)

def build_model(cfg):
    n_out = 1 + (N_REP if cfg["REPLICATE_HEADS"] else 0)
    return InitiationNetV2(cfg["CH_LOCAL"], tuple(cfg["TOWER"]), cfg["STRIDE"], cfg["ATTN_LAYERS"], cfg["ATTN_HEADS"], n_out)

In [ ]:
# 5.2 Визуализация архитектуры: слои, размеры тензоров, параметры, рецептивное поле
net = build_model(CFG).to(DEVICE).eval()
n_params = lambda m: sum(p.numel() for p in m.parameters())

# --- размеры выходов каждого блока (прогоняем одно окно с хуками) ---
parts = [("stem", net.stem), ("local", net.local)]
for i in range(len(CFG["TOWER"])): parts += [(f"down{i + 1}", net.downs[i]), (f"blocks{i + 1}", net.blocks[i])]
parts += [("attention", net.attn)] + [(f"up{i + 1}", net.ups[i]) for i in range(len(CFG["TOWER"]))] + [("head", net.head)]
shapes = {}
hooks = [m.register_forward_hook(lambda m, i, o, n=n: shapes.__setitem__(n, tuple(o.shape))) for n, m in parts]
with torch.no_grad(): net(torch.zeros(1, 6, CFG["WIN"], device=DEVICE))
for h in hooks: h.remove()
arch = pd.DataFrame([{"блок": n, "выход (B, каналы, длина)": str(shapes.get(n, "-")), "параметров": n_params(m),
                      "разрешение, нт": CFG["WIN"] // shapes[n][-1] if n in shapes else "-"} for n, m in parts])
print(f"всего параметров: {n_params(net):,}  ({n_params(net) * 4 / 1e6:.1f} МБ в float32)")
display(arch)

# --- аналитическое рецептивное поле по уровням ---
def rf_levels(cfg):
    rf, jump, out = 1, 1, {}
    def conv(k, d=1, s=1):
        nonlocal rf, jump
        rf += (k - 1) * d * jump; jump *= s
    conv(11); [conv(5, 1) for _ in range(2)]; [conv(5, 2) for _ in range(2)]; out["local"] = rf
    for i in range(len(cfg["TOWER"])):
        conv(2 * cfg["STRIDE"] + 1, 1, cfg["STRIDE"]); [conv(5, 1) for _ in range(2)]; [conv(5, 3) for _ in range(2)]
        out[f"уровень {i + 1}\n(шаг {jump} нт)"] = rf
    if cfg["ATTN_LAYERS"] > 0: out["attention"] = cfg["WIN"]
    return out
rfs = rf_levels(CFG)

# --- эффективное рецептивное поле: |градиент выхода в центре окна| по входу (сеть со случайными весами) ---
def effective_rf(model, n=4):
    model.eval(); W = CFG["WIN"]
    x = encode(torch.randint(0, 4, (n, W), device=DEVICE)).requires_grad_(True)
    model(x)[:, 0, W // 2].sum().backward()
    g = x.grad.abs().sum(1).mean(0).detach().cpu().numpy()
    return np.arange(W) - W // 2, g / g.max()

fig = plt.figure(figsize=(15, 8)); gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])
axd = fig.add_subplot(gs[0, :]); axd.axis("off"); axd.set_xlim(0, 15); axd.set_ylim(-len(CFG["TOWER"]) - 1.6, 1.2)
def box(x, y, text, color, w=2.2, h=0.75):
    axd.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.05", fc=color, ec="k", lw=.8))
    axd.text(x, y, text, ha="center", va="center", fontsize=8)
def arrow(x0, y0, x1, y1, txt=""):
    axd.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="->", lw=.9))
    if txt: axd.text((x0 + x1) / 2, (y0 + y1) / 2 + .1, txt, fontsize=7, ha="center", color="dimgray")
W = CFG["WIN"]
box(1.2, 0, f"вход\n6 × {W}\nACGT+регистр+N", "#e8e8e8")
box(3.9, 0, f"stem: conv11\n{CFG['CH_LOCAL']} × {W}", "#cfe2f3")
box(6.6, 0, f"локальная ветка\n2 ResBlock (d=1,2)\n{CFG['CH_LOCAL']} × {W}", "#cfe2f3")
box(10.2, 0, "Σ слияние\nвсех масштабов", "#fff2cc", w=1.8)
box(12.6, 0, f"голова 1×1\n→ {1 + (N_REP if CFG['REPLICATE_HEADS'] else 0)} log-rate", "#d9ead3", w=2.0)
arrow(2.3, 0, 2.8, 0); arrow(5.0, 0, 5.5, 0); arrow(7.7, 0, 9.3, 0, "+ сама ветка"); arrow(11.1, 0, 11.6, 0)
prev = (6.6, -0.38)
for i, ch in enumerate(CFG["TOWER"]):
    y = -(i + 1); stride = CFG["STRIDE"] ** (i + 1)
    box(6.6, y, f"уровень {i + 1}: conv stride {CFG['STRIDE']} + 2 ResBlock\n{ch} × {W // stride} (шаг {stride} нт)", "#f4cccc", w=3.6)
    arrow(prev[0], prev[1], 6.6, y + .38); prev = (6.6, y - .38)
    axd.annotate("", (10.2, -0.38), (8.4, y), arrowprops=dict(arrowstyle="->", lw=.7, color="tab:red", connectionstyle="arc3,rad=-0.15"))
    axd.text(9.0, y + .15, f"1×1 → {CFG['CH_LOCAL']}, растянуть ×{stride}", fontsize=7, color="tab:red")
if CFG["ATTN_LAYERS"] > 0:
    y = -len(CFG["TOWER"]) - 1.1
    box(6.6, y, f"НОВОЕ: attention × {CFG['ATTN_LAYERS']} ({CFG['ATTN_HEADS']} голов)\nна {W // CFG['STRIDE'] ** len(CFG['TOWER'])} токенах уровня {len(CFG['TOWER'])}", "#d9d2e9", w=3.6)
    axd.annotate("", (4.8, y), (4.8, -len(CFG["TOWER"])), arrowprops=dict(arrowstyle="<->", lw=.7, color="purple"))
    axd.text(3.2, y + .45, "применяется к выходу\nпоследнего уровня", fontsize=7, color="purple")
axd.text(12.6, -1.2, "минус-цепь = та же сеть\nна обратно-комплементарном\nокне (выход развёрнут)", fontsize=8, ha="center", style="italic")
axd.set_title("Схема InitiationNetV2", fontsize=11)

ax1 = fig.add_subplot(gs[1, 0])
pp = arch.set_index("блок")["параметров"]
ax1.barh(pp.index[::-1], pp.values[::-1] / 1e3, color=["#d9d2e9" if n == "attention" else "#9fc5e8" for n in pp.index[::-1]])
ax1.set(xlabel="тыс. параметров", title=f"Параметры по блокам (всего {n_params(net) / 1e6:.2f} млн)")
ax2 = fig.add_subplot(gs[1, 1])
ax2.bar(list(rfs), list(rfs.values()), color="tab:purple"); ax2.set_yscale("log")
ax2.axhline(W, ls="--", color="k", lw=1); ax2.text(0, W * 1.1, f"окно WIN = {W}", fontsize=8)
ax2.set(ylabel="нт (лог. шкала)", title="Рецептивное поле после каждого уровня"); ax2.tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()

off, g = effective_rf(net)
fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(off, g, lw=.6); ax.set_yscale("log"); ax.set(xlabel="смещение от предсказываемой позиции, нт", ylabel="|градиент| (норм.)",
                                                      title="Эффективное рецептивное поле (сеть до обучения): какие нуклеотиды окна влияют на предсказание в центре")
plt.tight_layout(); plt.show()
del net

## 6. Обучение

* **Оптимизатор** AdamW, расписание OneCycle (разогрев `WARMUP`, затем косинусный спад), обрезка градиента 1.0.
* **Смешанная точность**: bfloat16 на A100/L4; float16 + GradScaler на T4. Сам Poisson-loss всегда в float32 (иначе `exp` переполняется:
  в одной позиции бывает до ~7.7·10⁵ прочтений). Шаги с нечисловым loss/градиентом пропускаются и считаются.
* **EMA**: параллельно хранится сглаженная копия весов `θ_ema ← d·θ_ema + (1−d)·θ`; именно её оцениваем и сохраняем.
* Каждые `EVAL_EVERY` шагов: быстрая проверка на группе **tune** (AUPRC и Spearman по сырым log-rate), сохраняется лучший шаг.
* Чекпоинты пишутся в `OUT_DIR`; если чекпоинт уже есть, обучение пропускается (удобно после разрыва сессии Colab).

In [ ]:
# 6.1 Метрики (точная копия официального скорера) и rank-power квантование
SCALE = 100_000                          # в сабмите 5 знаков после запятой -> 100 001 уровень

def quantise(pred):
    return np.rint(np.clip(pred, 0.0, 1.0) * SCALE).astype(np.int32)

def prauc_from_levels(levels, labels):
    # average precision по гистограмме уровней (то же, что sklearn с группировкой равных значений), без сортировки
    pos = np.bincount(levels[labels], minlength=SCALE + 1)[::-1].astype(np.int64)
    tot = np.bincount(levels, minlength=SCALE + 1)[::-1].astype(np.int64)
    keep = tot > 0; tp, n = np.cumsum(pos)[keep], np.cumsum(tot)[keep]
    P = tp[-1] if len(tp) else 0
    if P == 0: return 0.0
    return float((np.diff(np.r_[0, tp]) / P * (tp / n)).sum())

def dense_rank(x):
    return np.unique(x, return_inverse=True)[1].astype(np.float64)

def spearman_dense(pred_pos, gt_pos):
    if len(pred_pos) < 2: return 0.0
    a, b = dense_rank(pred_pos), dense_rank(gt_pos)
    a -= a.mean(); b -= b.mean(); d = np.sqrt((a * a).sum() * (b * b).sum())
    return 0.0 if d == 0 else float(a @ b / d)

def official_score(pred01, gt, mask=None):
    # pred01 — значения сабмита в [0, 1]; считаем ровно так, как скорер считает по файлу
    if mask is not None: pred01, gt = pred01[mask], gt[mask]
    lv, y = quantise(pred01), gt > 0
    return {"AUPRC": prauc_from_levels(lv, y), "Spearman": spearman_dense(lv[y], gt[y])}

def raw_score(lograte, gt, mask=None):
    # метрики по «сырому» предсказанию без квантования (для контроля обучения)
    if mask is not None: lograte, gt = lograte[mask], gt[mask]
    y = gt > 0
    return {"AUPRC": float(average_precision_score(y, lograte)), "Spearman": spearman_dense(lograte[y], gt[y])}

def rank_power(lograte, alpha, nmask=None):
    # монотонное преобразование: место в рейтинге r (1 = самое сильное) -> r^(-alpha), округление до 5 знаков; N -> 0
    x = lograte.astype(np.float32, copy=True)
    if nmask is not None: x[nmask] = -np.inf
    order = np.argsort(-x, kind="stable")
    ranks = np.empty(len(x), np.int32); ranks[order] = np.arange(1, len(x) + 1, dtype=np.int32); del order
    s = (1.0 - (ranks - 1) / len(x)) if alpha == 0 else np.power(ranks.astype(np.float32), -alpha)
    s = np.rint(np.clip(s, 0, 1) * SCALE).astype(np.float32) / SCALE
    if nmask is not None: s[nmask] = 0.0
    return s

In [ ]:
# 6.2 Функция потерь, EMA и быстрая проверка на группе tune
def poisson_nll(log_rate, counts, mask):
    lam = log_rate.float().clamp(-20, 15)                       # float32 обязательно
    return -((counts * lam - torch.exp(lam)) * mask).sum() / mask.sum().clamp(min=1.0)

def compute_loss(out, y, mask, cfg):
    # out, y: (B, K, 2, L); mask: (B, L). Считаем только в центральной зоне окна
    c = cfg["CROP"]; m = mask[:, None, c:-c].expand(-1, 2, -1)
    losses = [poisson_nll(out[:, k, :, c:-c], y[:, k, :, c:-c], m) for k in range(out.shape[1])]
    total = losses[0] + (cfg["AUX_WEIGHT"] * torch.stack(losses[1:]).mean() if len(losses) > 1 else 0.0)
    return total, losses

class EMA:
    def __init__(self, model, decay):
        self.decay, self.model = decay, copy.deepcopy(model).eval()
        for p in self.model.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))           # в начале EMA быстрее догоняет модель
        for e, p in zip(self.model.parameters(), model.parameters()): e.mul_(d).add_(p.detach(), alpha=1 - d)
        for e, b in zip(self.model.buffers(), model.buffers()): e.copy_(b)

def make_eval_set(contigs, cfg, seed=0):
    # непересекающиеся окна по whitelist контигов группы; берём случайные, пока не наберём EVAL_BP нт центральных зон
    W, C = cfg["WIN"], cfg["CROP"]; step = W - 2 * C; rng = np.random.default_rng(seed)
    starts = [(c, s) for c in contigs for s in range(-C, len(genome[c]) - C, step) if IN_WL[c][max(s + C, 0):s + C + step].any()]
    starts = [starts[i] for i in rng.permutation(len(starts))[: max(1, cfg["EVAL_BP"] // step)]]
    codes = np.stack([window_codes(c, s, W) for c, s in starts])
    ys, ms = zip(*[window_targets(c, s, W) for c, s in starts])
    y = np.stack(ys)[:, 0, :, C:W - C]                          # сумма реплик, (n, 2, step)
    m = (np.stack(ms)[:, None, C:W - C] > 0) & (codes[:, None, C:W - C] != 8)
    return dict(codes=codes, y=y, mask=np.broadcast_to(m, y.shape))

@torch.no_grad()
def predict_windows(models, codes, cfg):
    # средний log-rate суммы реплик по моделям: (n, 2, WIN)
    out = []
    for i in range(0, len(codes), cfg["PRED_BATCH"]):
        x = encode(torch.from_numpy(codes[i:i + cfg["PRED_BATCH"]]).to(DEVICE), cfg["USE_LOWERCASE"])
        acc = 0
        for net in models:
            net.eval()
            with autocast(): acc = acc + forward_both(net, x)[:, 0].float()
        out.append((acc / len(models)).cpu().numpy())
    return np.concatenate(out)

def quick_eval(models, ev, cfg):
    C, W = cfg["CROP"], cfg["WIN"]
    lr = predict_windows(models, ev["codes"], cfg)[:, :, C:W - C]
    return raw_score(lr[ev["mask"]], ev["y"][ev["mask"]])

EVAL_SET = make_eval_set(GROUPS["tune"], CFG)
print(f"быстрая проверка: {len(EVAL_SET['codes'])} окон, позиций {int(EVAL_SET['mask'].sum()):,}, положительных {int((EVAL_SET['y'][EVAL_SET['mask']] > 0).sum()):,}")

In [ ]:
# 6.3 Обучение одной модели
def train_model(cfg, seed, tag="v2"):
    path = f"{OUT_DIR}/{tag}_seed{seed}.pt"
    if os.path.exists(path):
        ck = torch.load(path, map_location=DEVICE, weights_only=False)
        net = build_model(ck["cfg"]).to(DEVICE); net.load_state_dict(ck["model"]); net.eval()
        print("загружена готовая модель", path); return net, ck["hist"], ck["evals"]

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    net = build_model(cfg).to(DEVICE)
    ema = EMA(net, cfg["EMA_DECAY"])
    opt = torch.optim.AdamW(net.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["LR"], total_steps=cfg["STEPS"], pct_start=cfg["WARMUP"])
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    sampler = WindowSampler(FIT_CONTIGS, cfg, seed)
    hist, evals, best, best_state, bad = [], [], -1.0, None, 0

    pbar = tqdm(range(1, cfg["STEPS"] + 1), desc=f"{tag} seed {seed}", unit="шаг", mininterval=2)
    for step in pbar:
        net.train()
        codes, y, mask = (t.to(DEVICE, non_blocking=True) for t in sampler.batch(cfg["BATCH"]))
        if not cfg["REPLICATE_HEADS"]: y = y[:, :1]
        with autocast():
            out = forward_both(net, encode(codes, cfg["USE_LOWERCASE"]))
        loss, parts = compute_loss(out, y, mask, cfg)
        opt.zero_grad(set_to_none=True)
        if torch.isfinite(loss):
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            if torch.isfinite(gn): scaler.step(opt); ema.update(net, step)
            else: bad += 1
            scaler.update()
        else:
            bad += 1; gn = torch.tensor(float("nan"))
        sched.step()
        if step % 10 == 0 or step == 1:
            hist.append(dict(step=step, loss=loss.item(), loss_main=parts[0].item(),
                             loss_reps=torch.stack(parts[1:]).mean().item() if len(parts) > 1 else np.nan,
                             lr=sched.get_last_lr()[0], grad_norm=float(gn), bad_steps=bad))
            pbar.set_postfix(loss=f"{hist[-1]['loss_main']:.4f}", best_AUPRC=f"{best:.4f}", bad=bad)
        if step % cfg["EVAL_EVERY"] == 0 or step == cfg["STEPS"]:
            r_raw, r_ema = quick_eval([net], EVAL_SET, cfg), quick_eval([ema.model], EVAL_SET, cfg)
            evals.append(dict(step=step, AUPRC_raw=r_raw["AUPRC"], AUPRC_ema=r_ema["AUPRC"], Spearman_raw=r_raw["Spearman"], Spearman_ema=r_ema["Spearman"]))
            tqdm.write(f"шаг {step}: AUPRC tune {r_raw['AUPRC']:.4f} (EMA {r_ema['AUPRC']:.4f}), Spearman {r_raw['Spearman']:.3f} (EMA {r_ema['Spearman']:.3f})")
            if r_ema["AUPRC"] > best:
                best, best_state = r_ema["AUPRC"], copy.deepcopy(ema.model.state_dict())
    if best_state is None: best_state = copy.deepcopy(ema.model.state_dict())
    ema.model.load_state_dict(best_state)
    torch.save(dict(model=best_state, cfg=dict(cfg), hist=hist, evals=evals, seed=seed), path)
    print(f"сохранено {path}; лучший AUPRC tune (EMA) {best:.4f}; пропущено шагов {bad}")
    return ema.model.eval(), hist, evals

def plot_training(runs):
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.3))
    for name, (h, e) in runs.items():
        h, e = pd.DataFrame(h), pd.DataFrame(e)
        ax[0].plot(h.step, h.loss_main.rolling(50, min_periods=1).mean(), label=f"{name}: сумма реплик")
        if h.loss_reps.notna().any(): ax[0].plot(h.step, h.loss_reps.rolling(50, min_periods=1).mean(), ls=":", label=f"{name}: реплики")
        ax[1].plot(h.step, h.lr)
        ax[2].plot(h.step, h.grad_norm.rolling(50, min_periods=1).mean(), label=name)
        ax[3].plot(e.step, e.AUPRC_raw, "o--", alpha=.6, label=f"{name}: веса"); ax[3].plot(e.step, e.AUPRC_ema, "o-", label=f"{name}: EMA")
    ax[0].set(title="Poisson loss (скользящее среднее)", xlabel="шаг"); ax[0].legend(fontsize=7)
    ax[1].set(title="Скорость обучения (OneCycle)", xlabel="шаг")
    ax[2].set(title="Норма градиента (до обрезки 1.0)", xlabel="шаг"); ax[2].set_yscale("log")
    ax[3].set(title="AUPRC на tune (без квантования)", xlabel="шаг"); ax[3].legend(fontsize=7)
    plt.tight_layout(); plt.show()

In [ ]:
# 6.4 Обучаем ансамбль (N_SEEDS моделей). На L4 одна модель Nematostella ~ несколько часов; прогресс-бар показывает прогноз
MODELS, RUNS = [], {}
for seed in range(CFG["N_SEEDS"]):
    net, hist, evals = train_model(CFG, seed)
    MODELS.append(net); RUNS[f"seed {seed}"] = (hist, evals)
plot_training(RUNS)
display(pd.concat({k: pd.DataFrame(e).set_index("step") for k, (h, e) in RUNS.items()}, axis=1).round(4))
print("Как читать: loss должен падать и выходить на плато; AUPRC на tune расти. EMA-кривая обычно выше и глаже кривой «сырых» весов.")

In [ ]:
# 6.5 Что выучил первый слой: фильтры stem (ширина 11) как матрицы «основание × позиция»
W_stem = MODELS[0].stem[0].weight.detach().float().cpu().numpy()      # (каналы, 6, 11)
order = np.argsort(-np.linalg.norm(W_stem[:, :4], axis=(1, 2)))[:16]
fig, axs = plt.subplots(2, 8, figsize=(16, 3.6))
for a, k in zip(axs.ravel(), order):
    w = W_stem[k, :4] - W_stem[k, :4].mean(0, keepdims=True)           # вклад основания относительно среднего в колонке
    v = np.abs(w).max(); a.imshow(w, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
    a.set_yticks(range(4), list("ACGT")); a.set_xticks([]); a.grid(False)
    a.set_title(f"#{k}: " + "".join("ACGT"[i] if w[i, j] > .5 * v else "." for j, i in enumerate(w.argmax(0))), fontsize=8, family="monospace")
fig.suptitle("16 самых «сильных» фильтров первой свёртки (красный = основание повышает выход фильтра). В заголовке: консенсус", fontsize=10)
plt.tight_layout(); plt.show()
fl = np.abs(W_stem[:, 4:]).mean((0, 2)); fb = np.abs(W_stem[:, :4]).mean()
print(f"средний |вес|: каналы ACGT {fb:.3f}, флаг строчной буквы {fl[0]:.3f}, флаг N {fl[1]:.3f} (если флаги ~0, модель их не использует)")

## 7. Предсказание и оценка на отложенных контигах

**Предсказание контига.** Окна `WIN` идут с шагом `WIN − 2·CROP`; от каждого окна берём только центральную зону.
Нарезка повторяется `TTA_SHIFTS` раз со сдвигом на долю шага, результаты (и все модели ансамбля) усредняются по log-rate.
Окна, центральная зона которых не задевает ни одного интервала, пропускаются (на `Ofas_2.0` много N и мелких контигов).

**Квантование.** Сабмит хранит 5 знаков, т.е. 100 001 уровень. AUPRC хочет мелких уровней, Spearman по dense-рангам — крупных
(у правды всего несколько десятков различных значений). Поэтому место в рейтинге `r` превращаем в `r^(−alpha)`:
верх рейтинга получает много уровней, хвост сливается. `alpha` выбираем на **tune**: наибольшее, при котором AUPRC ≥ `KEEP_FRAC`·лучшего.

In [ ]:
# 7.1 Строки шаблона, предсказание по контигам, сбор в порядке шаблона
def make_rows(df):
    # df: chrom, start, end, strand ('+'/'-' или 0/1) -> добавляем длину и смещение строки в итоговом векторе
    r = df[["chrom", "start", "end"]].copy()
    st = df["strand"]
    r["strand"] = st.values.astype(int) if pd.api.types.is_integer_dtype(st) else (st.astype(str).values == "-").astype(int)
    r["len"] = r.end - r.start; r["off"] = np.r_[0, np.cumsum(r.len.values)[:-1]]
    return r.reset_index(drop=True)

def train_rows(contigs):
    iv = wl_train[wl_train.chrom.isin(set(contigs))]
    return make_rows(pd.concat([iv.assign(strand=0), iv.assign(strand=1)]))

def gather_rows(rows, contig_fn, dtype=np.float32, desc=None):
    # contig_fn(c, rows_c, lo, hi) -> массив (2, hi-lo); раскладываем его в порядок строк
    out = np.zeros(int(rows.len.sum()), dtype)
    groups = rows.groupby("chrom", sort=False)
    for c, g in tqdm(groups, total=groups.ngroups, desc=desc, disable=desc is None, mininterval=2):
        lo, hi = int(g.start.min()), int(g.end.max()); arr = contig_fn(c, g, lo, hi)
        for s, e, st, o in zip(g.start.values, g.end.values, g.strand.values, g.off.values):
            out[o:o + e - s] = arr[st, s - lo:e - lo]
    return out

def rows_nmask(rows):
    return gather_rows(rows, lambda c, g, lo, hi: np.broadcast_to(genome[c][lo:hi] == 8, (2, hi - lo)), bool)

def rows_targets(rows):
    def fn(c, g, lo, hi):
        a = np.zeros((2, hi - lo), np.float32); t = TRACK.get(c)
        if t is not None:
            k = (t["pos"] >= lo) & (t["pos"] < hi); a[t["strand"][k], t["pos"][k] - lo] = t["tot"][k]
        return a
    return gather_rows(rows, fn)

@torch.no_grad()
def predict_contig(models, c, g, lo, hi, cfg):
    W, C = cfg["WIN"], cfg["CROP"]; step = W - 2 * C
    iv = np.unique(np.stack([g.start.values, g.end.values], 1), axis=0)          # строки по возрастанию start
    ms, me = iv[:, 0], np.maximum.accumulate(iv[:, 1])
    def overlaps(a, b):                                          # задевает ли [a, b) хоть один интервал
        k = np.searchsorted(me, a, side="right"); return k < len(ms) and ms[k] < b
    starts = [s for k in range(cfg["TTA_SHIFTS"]) for s in range(lo - C - k * step // cfg["TTA_SHIFTS"], hi - C, step) if overlaps(s + C, s + C + step)]
    acc = np.zeros((2, hi - lo), np.float32); cnt = np.zeros(hi - lo, np.float32)
    for i in range(0, len(starts), cfg["PRED_BATCH"]):
        chunk = starts[i:i + cfg["PRED_BATCH"]]
        lr = predict_windows(models, np.stack([window_codes(c, s, W) for s in chunk]), cfg)[:, :, C:W - C]
        for s, p in zip(chunk, lr):
            a, b = max(s + C, lo), min(s + C + step, hi)
            if b > a: acc[:, a - lo:b - lo] += p[:, a - s - C:b - s - C]; cnt[a - lo:b - lo] += 1
    return acc / np.maximum(cnt, 1)

def predict_rows(models, rows, cfg=CFG, desc="предсказание"):
    return gather_rows(rows, lambda c, g, lo, hi: predict_contig(models, c, g, lo, hi, cfg), desc=desc)

In [ ]:
# 7.2 Предсказание групп tune и report: каждая модель отдельно и ансамбль; подбор alpha на tune; честная оценка на report
HO = {}
for gname in ("tune", "report"):
    rows = train_rows(GROUPS[gname])
    HO[gname] = dict(rows=rows, gt=rows_targets(rows), n=rows_nmask(rows),
                     preds=[predict_rows([m], rows, desc=f"{gname}, модель {i}") for i, m in enumerate(MODELS)])
    HO[gname]["ens"] = np.mean(HO[gname]["preds"], axis=0)

def alpha_grid(lograte, gt, nmask, alphas):
    sel = ~nmask
    return pd.DataFrame([{"alpha": a, **official_score(rank_power(lograte[sel], a), gt[sel])} for a in alphas]).set_index("alpha")

grid = alpha_grid(HO["tune"]["ens"], HO["tune"]["gt"], HO["tune"]["n"], CFG["ALPHAS"])
grid["AUPRC/лучший"] = grid.AUPRC / grid.AUPRC.max()
ALPHA = float(grid.index[grid["AUPRC/лучший"] >= CFG["KEEP_FRAC"]].max())
print(f"выбран alpha = {ALPHA} (наибольший, при котором AUPRC на tune >= {CFG['KEEP_FRAC']} от лучшего)")
display(grid.round(5))

def evaluate(lograte, gt, nmask, alpha):
    full = rank_power(lograte, alpha, nmask)
    return {**{f"{k} (сырое)": v for k, v in raw_score(lograte, gt, ~nmask).items()}, **official_score(full, gt)}

rep = HO["report"]
table = pd.DataFrame({**{f"модель seed {i}": evaluate(p, rep["gt"], rep["n"], ALPHA) for i, p in enumerate(rep["preds"])},
                      f"ансамбль ({len(MODELS)})": evaluate(rep["ens"], rep["gt"], rep["n"], ALPHA),
                      "ансамбль, alpha=0 (без настройки квантования)": evaluate(rep["ens"], rep["gt"], rep["n"], 0.0)}).T
REPORT_TABLE = table
display(table.round(4))
print(f"report: позиций {len(rep['gt']):,}, положительных {int((rep['gt'] > 0).sum()):,}, доля {float((rep['gt'] > 0).mean()):.2e} (случайный AUPRC)")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.3))
ax[0].plot(grid.index, grid["AUPRC/лучший"], "o-", color="tab:blue"); ax[0].axhline(CFG["KEEP_FRAC"], ls="--", color="gray")
ax[0].axvline(ALPHA, color="tab:red", lw=1); ax[0].set(xlabel="alpha", ylabel="AUPRC / лучший", title="tune: цена квантования для AUPRC")
ax[1].plot(grid.index, grid.Spearman, "o-", color="tab:green"); ax[1].axvline(ALPHA, color="tab:red", lw=1, label=f"выбран {ALPHA}")
ax[1].set(xlabel="alpha", ylabel="Spearman", title="tune: выигрыш Spearman"); ax[1].legend()
t = table.iloc[:-1]
x = np.arange(len(t)); ax[2].bar(x - .2, t.AUPRC, .4, label="AUPRC"); a2 = ax[2].twinx(); a2.bar(x + .2, t.Spearman, .4, color="tab:green", label="Spearman")
ax[2].set_xticks(x, t.index, rotation=15, fontsize=8); ax[2].set_title("report: отдельные модели и ансамбль"); ax[2].set_ylabel("AUPRC"); a2.set_ylabel("Spearman")
ax[2].legend(loc="upper left"); a2.legend(loc="upper right"); a2.grid(False)
plt.tight_layout(); plt.show()

## 8. Тест: предсказание, офлайн-оценка (только Nematostella) и сравнение с бейзлайном

In [ ]:
# 8.1 Предсказание теста ансамблем (в порядке template.test.bed) и офлайн-оценка
TEST_ROWS = make_rows(tpl_test)
TEST_N = rows_nmask(TEST_ROWS)
TEST_PRED = predict_rows(MODELS, TEST_ROWS, desc="тест")
np.save(f"{OUT_DIR}/test_lograte.npy", TEST_PRED)              # сырые log-rate: alpha можно переподобрать без GPU
print(f"тест: {len(TEST_PRED):,} позиций, из них N: {int(TEST_N.sum()):,} ({TEST_N.mean():.2%})")

HAS_GT = os.path.exists(f"{BASE}/ground_truth/csRNA-test.txt.gz")
read_col = lambda p, dt: pd.read_csv(p, header=None, dtype=dt)[0].values
OURS = {}
if HAS_GT:
    gt = read_col(f"{BASE}/ground_truth/csRNA-test.txt.gz", np.float32)
    m1 = read_col(f"{BASE}/scoring/mask1-test.txt.gz", np.int8).astype(bool); m2 = read_col(f"{BASE}/scoring/mask2-test.txt.gz", np.int8).astype(bool)
    assert len(gt) == len(TEST_PRED)
    sub = rank_power(TEST_PRED, ALPHA, TEST_N)
    res = pd.DataFrame({nm: official_score(sub, gt, mk) for nm, mk in (("half1", m1), ("half2", m2), ("весь тест", np.ones_like(m1)))}).T
    display(res.round(4))
    OURS = {"AUPRC": float(res.loc[["half1", "half2"], "AUPRC"].mean()), "Spearman": float(res.loc[["half1", "half2"], "Spearman"].mean())}
    REF_NAME = "официальный тест, среднее half1/half2"
else:
    OURS = {"AUPRC": float(REPORT_TABLE.iloc[-2]["AUPRC"]), "Spearman": float(REPORT_TABLE.iloc[-2]["Spearman"])}
    REF_NAME = "hold-out контиги train (группы у бейзлайна свои, сравнение примерное)"

BASELINE_REF = {
    "jaNemVect1.1": {"динуклеотиды (организаторы)": (0.00116, 0.0361), "прошлый пайплайн*": (0.0434, 0.379),
                     "InitiationNet wide 24k": (0.11894, 0.3749), "InitiationNet big 96k": (0.13165, 0.391)},
    "Ofas_2.0": {"InitiationNet wide 96k": (0.15914, 0.4436)},
}
ref = BASELINE_REF.get(SPECIES, {})
cmp_tab = pd.DataFrame({**{k: {"AUPRC": a, "Spearman": s} for k, (a, s) in ref.items()}, "НАША v2": OURS}).T
display(cmp_tab.round(4)); print("оценка:", REF_NAME, "| * прошлый пайплайн оценён без квантования и обычным Spearman")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for a, met in zip(ax, ("AUPRC", "Spearman")):
    cols = ["tab:red" if i == "НАША v2" else "tab:gray" for i in cmp_tab.index]
    a.barh(cmp_tab.index, cmp_tab[met], color=cols); a.set_title(f"{met}: {SPECIES}")
    for i, v in enumerate(cmp_tab[met]): a.text(v, i, f" {v:.3f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 8.2 Картинки результата: PR-кривая, предсказание против правды, участок генома, метагенный профиль, saliency
rep = HO["report"]; sel = ~rep["n"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for nm, p in [(f"seed {i}", q) for i, q in enumerate(rep["preds"])] + [("ансамбль", rep["ens"])]:
    pr, rc_, _ = precision_recall_curve(rep["gt"][sel] > 0, p[sel]); k = max(1, len(pr) // 4000)
    ax[0].plot(rc_[::k], pr[::k], lw=2 if nm == "ансамбль" else 1, label=nm)
ax[0].axhline(float((rep["gt"][sel] > 0).mean()), ls="--", color="gray", label="случайный уровень")
ax[0].set(xlabel="recall", ylabel="precision", title="PR-кривая (report)", yscale="log"); ax[0].legend(fontsize=8)
pos = rep["gt"] > 0
bins = pd.cut(rep["gt"][pos], [0, 1, 2, 4, 9, 49, np.inf], labels=["1", "2", "3-4", "5-9", "10-49", "50+"])
data = [rep["ens"][pos][bins == b] for b in bins.categories]
ax[1].boxplot(data, showfliers=False); ax[1].set_xticks(range(1, len(data) + 1), bins.categories)
ax[1].set(xlabel="настоящий счёт (сумма реплик)", ylabel="предсказанный log-rate", title="Сила: предсказание растёт с настоящим счётом?")
plt.tight_layout(); plt.show()

# --- участок генома и метагенный профиль по крупнейшему контигу группы report ---
c = max(GROUPS["report"], key=lambda x: wl_bp[x]); t = TRACK[c]
g = train_rows([c]); g = g[g.start < g.start.min() + 5_000_000]               # не больше 5 Мб, чтобы было быстро
lo, hi = int(g.start.min()), int(g.end.max())
P = np.exp(predict_contig(MODELS, c, g, lo, hi, CFG))                         # (2, hi-lo) ожидаемый счёт
true = np.zeros((2, hi - lo), np.float32); k = (t["pos"] >= lo) & (t["pos"] < hi); true[t["strand"][k], t["pos"][k] - lo] = t["tot"][k]
best = np.argmax(true.sum(0)); a, b = max(best - 1500, 0), min(best + 1500, hi - lo)
xs = np.arange(a, b) + lo
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].bar(xs, np.log1p(true[0, a:b]), width=2, color="tab:red", label="+ правда"); ax[0].bar(xs, -np.log1p(true[1, a:b]), width=2, color="tab:blue", label="− правда")
ax[0].fill_between(xs, -0.15, 0.15, where=IN_WL[c][a + lo:b + lo], color="tab:green", alpha=.3, label="whitelist"); ax[0].set_ylabel("ln(1+счёт)"); ax[0].legend(fontsize=8)
ax[1].plot(xs, P[0, a:b], color="tab:red", lw=.8, label="+ предсказание"); ax[1].plot(xs, -P[1, a:b], color="tab:blue", lw=.8, label="− предсказание")
ax[1].set(ylabel="ожидаемый счёт", xlabel=f"{c}, координата"); ax[1].legend(fontsize=8)
ax[0].set_title("Самый сильный участок report-контига: столбики правды и пики предсказания должны совпадать")
plt.tight_layout(); plt.show()

H = 300; strong = [(s, p - lo) for p, s, v in zip(t["pos"][k], t["strand"][k], t["tot"][k]) if v >= 5 and H <= p - lo < hi - lo - H]
prof = {nm: np.zeros(2 * H + 1) for nm in ("правда, та же цепь", "предсказание, та же цепь", "предсказание, другая цепь")}
for s, p in strong:
    sl = slice(p - H, p + H + 1); f = (lambda v: v) if s == 0 else (lambda v: v[::-1])   # для минус-цепи разворачиваем ось
    prof["правда, та же цепь"] += f(true[s, sl]); prof["предсказание, та же цепь"] += f(P[s, sl]); prof["предсказание, другая цепь"] += f(P[1 - s, sl])
fig, ax = plt.subplots(figsize=(11, 3))
for nm, v in prof.items(): ax.plot(np.arange(-H, H + 1), v / max(len(strong), 1), label=nm, lw=1)
ax.set(yscale="log", xlabel="смещение от сильной позиции (по направлению транскрипции), нт", ylabel="средний счёт",
       title=f"Метагенный профиль вокруг {len(strong)} позиций со счётом >= 5: ширина пика и дивергентный сигнал на другой цепи"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# --- saliency: градиент × вход вокруг самой сильной позиции; и эффективное рецептивное поле обученной сети ---
net = MODELS[0].eval(); W = CFG["WIN"]; p0 = best + lo; s0 = int(np.argmax(true[:, best]))
x = encode(torch.from_numpy(window_codes(c, p0 - W // 2, W)[None]).to(DEVICE), CFG["USE_LOWERCASE"])
if s0 == 1: x = rc(x)                                                       # для минус-цепи смотрим на RC-окно
x = x.clone().requires_grad_(True); net(x)[0, 0, W // 2].backward()
gi = (x.grad * x)[0, :4].detach().cpu().numpy(); sal = gi.sum(0)
base = x[0, :4].argmax(0).detach().cpu().numpy(); H2 = 40; sl = slice(W // 2 - H2, W // 2 + H2 + 1)
fig, ax = plt.subplots(2, 1, figsize=(13, 4.5))
ax[0].bar(np.arange(-H2, H2 + 1), sal[sl], color=np.array(["tab:green", "tab:blue", "orange", "tab:red"])[base[sl]])
for i, (v, bb) in enumerate(zip(sal[sl], base[sl])): ax[0].text(i - H2, v, "ACGT"[bb], ha="center", va="bottom" if v >= 0 else "top", fontsize=7)
ax[0].set(title="Saliency (градиент × вход) вокруг самой сильной позиции: какие нуклеотиды поднимают предсказание в точке 0", xlabel="смещение, нт")
off, gr = effective_rf(net)
ax[1].plot(off, gr, lw=.6); ax[1].set(yscale="log", xlabel="смещение от предсказываемой позиции, нт", ylabel="|градиент|", title="Эффективное рецептивное поле обученной сети")
plt.tight_layout(); plt.show()

In [ ]:
# 8.3 Файл сабмита: rank-power(alpha), N -> 0, 5 знаков, порядок template.test.bed; проверки формата
SUB = rank_power(TEST_PRED, ALPHA, TEST_N)
sub_path = f"{OUT_DIR}/{SPECIES}.predictions.txt.gz"
ints = np.rint(SUB * SCALE).astype(np.int64)
with gzip.open(sub_path, "wt", compresslevel=6) as fh:
    for a in tqdm(range(0, len(ints), 8_000_000), desc="пишу сабмит"):
        fh.write("\n".join(np.char.mod("%.5f", ints[a:a + 8_000_000] / SCALE))); fh.write("\n")
checks = {"строк = позиций шаблона": len(SUB) == TEST_LEN, "все значения в [0, 1]": bool((SUB >= 0).all() and (SUB <= 1).all()),
          "N-позиции = 0": bool((SUB[TEST_N] == 0).all()), "различных уровней": int(len(np.unique(ints))), "alpha": ALPHA,
          "размер файла, МБ": round(os.path.getsize(sub_path) / 1e6, 1)}
display(pd.Series(checks, name="проверка").to_frame())
assert checks["строк = позиций шаблона"] and checks["все значения в [0, 1]"]
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(SUB[SUB > 0], bins=np.logspace(-5, 0, 60)); ax.set(xscale="log", yscale="log", xlabel="значение в сабмите", ylabel="позиций", title="Распределение значений сабмита")
plt.tight_layout(); plt.show()
print("сабмит:", sub_path, "\nПеред загрузкой проверьте официальным валидатором: https://github.com/autosome-ru/EPIC_scoring")
# from google.colab import drive; drive.mount("/content/drive"); import shutil; shutil.copy(sub_path, "/content/drive/MyDrive/")

## 9. Абляция: что даёт каждое изменение

Обучаем по одной модели на каждый вариант (меньше шагов, чтобы влезло в сессию) и сравниваем на **report** с одним и тем же правилом выбора `alpha`.
Варианты включают изменения по очереди, начиная с конфигурации, повторяющей бейзлайн. Включите `RUN_ABLATIONS = True`;
на Nematostella это примерно 5 × `ABL_STEPS` шагов обучения.

In [ ]:
# 9.1 Абляция
RUN_ABLATIONS = FAST_DEV            # по умолчанию выключено (дорого); в FAST_DEV прогоняется для проверки кода
ABL_STEPS = 60 if FAST_DEV else 16000
EXPERIMENTS = {
    "как бейзлайн":         dict(ATTN_LAYERS=0, REPLICATE_HEADS=False, EMA_DECAY=0.0),
    "+ EMA":                dict(ATTN_LAYERS=0, REPLICATE_HEADS=False),
    "+ EMA + реплики":      dict(ATTN_LAYERS=0),
    "+ EMA + реплики + attention (v2)": dict(),
    "v2 без флага регистра": dict(USE_LOWERCASE=False),
}
if RUN_ABLATIONS:
    rows_t, rows_r = HO["tune"]["rows"], HO["report"]["rows"]
    abl = {}
    for i, (name, over) in enumerate(EXPERIMENTS.items()):
        cfg = {**CFG, "STEPS": ABL_STEPS, "EVAL_EVERY": max(ABL_STEPS // 4, 1), **over}
        if not N_REP >= 2: cfg["REPLICATE_HEADS"] = False
        m, _, _ = train_model(cfg, seed=0, tag=f"abl{i}")
        pt, pr = predict_rows([m], rows_t, cfg, desc=None), predict_rows([m], rows_r, cfg, desc=None)
        gt_ = alpha_grid(pt, HO["tune"]["gt"], HO["tune"]["n"], CFG["ALPHAS"])
        a_ = float(gt_.index[gt_.AUPRC >= CFG["KEEP_FRAC"] * gt_.AUPRC.max()].max())
        abl[name] = {"alpha": a_, **official_score(rank_power(pr, a_, HO["report"]["n"]), HO["report"]["gt"])}
        print(name, {k: round(v, 4) for k, v in abl[name].items()})
    abl = pd.DataFrame(abl).T; display(abl.round(4))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
    for a, met in zip(ax, ("AUPRC", "Spearman")):
        a.barh(abl.index[::-1], abl[met][::-1], color="tab:purple"); a.set_title(f"Абляция на report: {met}")
        for j, v in enumerate(abl[met][::-1]): a.text(v, j, f" {v:.4f}", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

## 10. Итоги и что дальше

**Как запустить для нашего вида.** В ячейке 1.2 поставьте `SPECIES = "Ofas_2.0"` (или переменную окружения `EPIC_SPECIES`), выберите GPU с High-RAM
и выполните все ячейки. Раздел 8.1 для Ofas_2.0 не считает тестовые метрики (ответов нет), а показывает hold-out `report` против бейзлайна.
Файл `Ofas_2.0.predictions.txt.gz` из раздела 8.3 загружается на https://epic.autosome.org.

**Порядок проверки улучшений.** Сначала Nematostella: раздел 8.1 даёт честное сравнение с бейзлайном на официальном тесте,
раздел 9 показывает вклад каждого изменения. Изменение, которое на Nematostella не помогает, на Ofas_2.0 стоит выключить.

**Что ещё можно попробовать (по ожидаемой пользе):**
1. Дообучить финальную модель на всех контигах, включая `tune`/`report` (сейчас они не участвуют в обучении, это ~10% данных).
2. Предобучение на другом насекомом из соревнования (моль `ilPloInte3.2`) с дообучением на `Ofas_2.0`: общий «ствол» сети, отдельные головы.
3. Больше шагов и моделей в ансамбле: у бейзлайна на Ofas_2.0 всего 9.6 эпох, кривая AUPRC на tune подскажет, есть ли запас.
4. Нефильтрованные треки (`*-unfiltered.bed.gz`) как дополнительные данные из повторов: маску loss тогда нужно расширить на весь контиг.